# Predict expression from H&E

Use DeepSpot-M to predict the Xenium gene panel and identify melanoma-enriched spots.


## Libraries


In [ ]:
from pathlib import Path
import json
import csv
import h5py
import numpy as np
import torch
from deepspotm import DeepSpotM


## Settings

Run from this folder and set the data and model paths in `config.json`. Leave `slide_to_run = None` to process all slides. Use `reencode = True` to generate image tokens again, or set `spot_limit` for a smaller run.


In [ ]:
root = Path.cwd()
config_path = root / 'config.json'
device = 'cuda'
slide_to_run = None
reencode = False
spot_limit = None


## Load the slides and gene panel


In [ ]:
config = json.loads(config_path.read_text())
data_dir = Path(config['workspace']) / 'virtualHE'
outdir = Path(config['output']) / 'virtualHE'
slides = [
    row for row in csv.DictReader(open(data_dir / 'inputs/slides.csv'))
    if not slide_to_run or row['slide'] == slide_to_run
]
if slide_to_run and not slides:
    raise ValueError('Slide not found in inputs/slides.csv')
folder = outdir / ('prediction_checks' if spot_limit else 'predictions')
folder.mkdir(parents=True, exist_ok=True)
genes = json.loads((root / 'inputs/panel.json').read_text())['supported_genes']
aliases = json.loads((root / 'inputs/gene_aliases.json').read_text())['aliases']


## Load DeepSpot-M

Keep the baseline decoder weights for expression prediction. The saved fine-tuned decoders are used for the melanoma-marker score.


In [ ]:
torch.set_num_threads(4)
model, transform = DeepSpotM.from_pretrained(str(data_dir / config['model']), source='scgpt', device=device)
model.freeze_backbone()
model.eval()
vocabulary = {g.upper(): i for i, g in enumerate(model.gene_names)}
parameters = dict(model.named_parameters())
baseline = {name: value.detach().cpu().clone() for name, value in parameters.items() if name.startswith('gene_decoder.')}


## Match gene names


In [ ]:
# Map gene names to the model vocabulary.
def indices(names):
    return torch.tensor([vocabulary[aliases.get(g, g).upper()] for g in names], device=device)


## Predict expression and select melanoma spots

For each slide, predict gene expression from image tokens and calculate the mean melanoma-marker score. Apply the threshold defined in `config.json` and save the predictions. Slides with an existing output file are skipped.


In [ ]:
for row in slides:
    slide = row['slide']
    target = folder / f'{slide}.npz'
    if target.exists():
        print(f'{slide}: existing output, skipping', flush=True)
        continue
    with np.load(data_dir / row['prediction']) as original, np.load(data_dir / row['additional']) as extra:
        centers = original['centers_xy']
        original_genes = original['genes'].astype(str).tolist()
        extra_genes = extra['genes'].astype(str).tolist()
        assert np.array_equal(centers, extra['centers_xy'])
    n = min(len(centers), spot_limit) if spot_limit else len(centers)
    cache = data_dir / row['tokens']
    if reencode:
        import zarr
        from PIL import Image
        cache = folder / f'{slide}_tokens.h5'
        image = zarr.open_array(str((data_dir / row['image']) / '0'), mode='r')
        width, height = round(112 / float(row['mpp_x'])), round(112 / float(row['mpp_y']))
        model.backbone.to(device)
        with h5py.File(cache, 'w') as h:
            h.create_dataset('centers_xy', data=centers[:n])
            for start in range(0, n, 8):
                patches = []
                for x, y in centers[start:min(start + 8, n)]:
                    x0, y0 = round(x - width / 2), round(y - height / 2)
                    patch = np.asarray(image[:, y0:y0 + height, x0:x0 + width]).transpose(1, 2, 0)
                    patches.append(transform(Image.fromarray(patch).resize((224, 224), Image.Resampling.LANCZOS)))
                with torch.no_grad():
                    tokens = model.backbone.forward_patch_tokens(torch.stack(patches).to(device)).cpu().numpy()
                if start == 0:
                    h.create_dataset('tokens', (n, *tokens.shape[1:]), dtype='f4', chunks=(1, *tokens.shape[1:]))
                h['tokens'][start:start + len(tokens)] = tokens
    model.backbone.cpu()
    torch.cuda.empty_cache()
    with h5py.File(cache) as h:
        if 'centers_xy' in h:
            assert np.array_equal(h['centers_xy'][:n], centers[:n])
        def decode(names):
            idx = indices(names)
            values = np.empty((n, len(names)), dtype='float32')
            for start in range(0, n, 16):
                stop = min(start + 16, n)
                tokens = h['tokens'][start:stop].astype('float32')
                if 'scales' in h:
                    tokens *= h['scales'][start:stop]
                with torch.no_grad():
                    values[start:stop] = model.gene_decoder(torch.from_numpy(tokens).to(device), need_weights=False, gene_indices=idx)[0].cpu().numpy()
            return values
        with torch.no_grad():
            for name, value in baseline.items():
                parameters[name].copy_(value)
        combined = np.concatenate([decode(original_genes), decode(extra_genes)], axis=1)
        combined_genes = original_genes + extra_genes
        expression = combined[:, [combined_genes.index(g) for g in genes]]
        marker_index = [original_genes.index(g) for g in config['marker_genes']]
        folds = []
        for checkpoint in config['checkpoints']:
            state = torch.load(data_dir / checkpoint, map_location='cpu', weights_only=True)
            with torch.no_grad():
                for name, value in state.items():
                    parameters[name].copy_(value)
            folds.append(decode(original_genes)[:, marker_index])
        markers = np.stack(folds).mean(axis=0)
    score = markers.mean(axis=1)
    assert np.isfinite(expression).all() and np.isfinite(score).all()
    np.savez_compressed(target, genes=np.array(genes), centers_xy=centers[:n], pretrained=expression,
                        marker_genes=np.array(config['marker_genes']), fine_tuned_marker_mean=markers,
                        marker_score=score, melanoma_mask=score >= config['threshold'])
    print(f'{slide}: {n} spots, {len(genes)} genes', flush=True)
